# Reshaping Data

## About the data
In this notebook, we will using daily temperature data from the [National Centers for Environmental Information (NCEI) API](https://www.ncdc.noaa.gov/cdo-web/webservices/v2). We will use the Global Historical Climatology Network - Daily (GHCND) dataset; see the documentation [here](https://www1.ncdc.noaa.gov/pub/data/cdo/documentation/GHCND_documentation.pdf).

This data was collected for New York City for October 2018, using the Boonton 1 station (GHCND:USC00280907). It contains:
- the daily minimum temperature (`TMIN`)
- the daily maximum temperature (`TMAX`)
- the daily temperature at time of observation (`TOBS`)

*Note: The NCEI is part of the National Oceanic and Atmospheric Administration (NOAA) and, as you can see from the URL for the API, this resource was created when the NCEI was called the NCDC. Should the URL for this resource change in the future, you can search for "NCEI weather API" to find the updated one.*

## Setup
We need to import `pandas` and read in the long format data to get started:

**한글 요약: 표 모양 바꾸기 4가지 (짝으로 기억)**

- long → wide: `pivot()` (인덱스가 1층일 때) / `unstack()` (인덱스가 여러 층일 때)
- wide → long: `melt()` (열을 녹여서 길게) / `stack()` (열을 인덱스로 내림)
- `.T` 는 행/열 뒤집기 (전치)
- MultiIndex(멀티인덱스) = 인덱스나 열이 2층 이상인 것. 처음엔 헷갈리는 게 정상. "바깥층 → 안쪽층" 순으로 접근

TMAX = 최고기온, TMIN = 최저기온, TOBS = 관측 시점 기온

In [ ]:
import pandas as pd

# long 형식 데이터 읽기 → 열 이름 바꾸기 → 열 추가, 를 체이닝으로 한 번에
# usecols → 3개 열만 읽고
# .rename(columns={'value': 'temp_C'}) → value를 temp_C로
# .assign(date=날짜 타입으로, temp_F=화씨 온도 새 열)
long_df = pd.read_csv(
    'data/long_data.csv', usecols=['date', 'datatype', 'value']
).rename(
    columns={'value': 'temp_C'}
).assign(
    date=lambda x: pd.to_datetime(x.date),
    temp_F=lambda x: (x.temp_C * 9/5) + 32
)
long_df.head()


,datatype,date,temp_C,temp_F
0,TMAX,2018-10-01,21.1,69.98
1,TMIN,2018-10-01,8.9,48.02
2,TOBS,2018-10-01,13.9,57.02
3,TMAX,2018-10-02,23.9,75.02
4,TMIN,2018-10-02,13.9,57.02


## Transposing
Transposing swaps the rows and the columns. We use the `T` attribute to do so:

In [ ]:
# .T → Transpose(전치). 행과 열을 뒤집음 (엑셀의 '행/열 바꿈' 붙여넣기와 같음)
# date를 인덱스로 하고 6행만 뽑아서 뒤집으면 → 날짜가 위쪽(열)으로, 항목이 왼쪽(행)으로
long_df.set_index('date').head(6).T


date,2018-10-01,2018-10-01,2018-10-01,2018-10-02,2018-10-02,2018-10-02
datatype,TMAX,TMIN,TOBS,TMAX,TMIN,TOBS
temp_C,21.10,8.90,13.90,23.90,13.90,17.20
temp_F,69.98,48.02,57.02,75.02,57.02,62.96


## Pivoting
Going from long to wide format.

### `pivot()`
We can restructure our data by picking a column to go in the index (`index`), a column whose unique values will become column names (`columns`), and the values to place in those columns (`values`). The `pivot()` method can be used when we don't need to perform any aggregation in addition to our restructuring (when our index is unique); if this is not the case, we need the `pivot_table()` method which we will cover in chapter 4.

In [ ]:
# .pivot() → long → wide로 바꾸는 명령어 (pivot = 축을 중심으로 돌리기)
# index='date' → 날짜를 행(인덱스)으로
# columns='datatype' → datatype의 값들(TMAX, TMIN, TOBS)을 각각 열 이름으로
# values='temp_C' → 칸에 채울 값은 temp_C
# → 날짜 1행 + TMAX/TMIN/TOBS 열 = wide 형식 완성!
pivoted_df = long_df.pivot(
    index='date', columns='datatype', values='temp_C'
)
pivoted_df.head()


datatype,TMAX,TMIN,TOBS
date,,,
2018-10-01,21.1,8.9,13.9
2018-10-02,23.9,13.9,17.2
2018-10-03,25.0,15.6,16.1
2018-10-04,22.8,11.7,11.7
2018-10-05,23.3,11.7,18.9


Now that the data is pivoted, we have wide format data that we can grab summary statistics with:

In [ ]:
# wide 형식이 되니 describe로 항목별 통계가 깔끔하게 나옴
pivoted_df.describe()


datatype,TMAX,TMIN,TOBS
count,31.000000,31.000000,31.000000
mean,16.829032,7.561290,10.022581
std,5.714962,6.513252,6.596550
min,7.800000,-1.100000,-1.100000
25%,12.750000,2.500000,5.550000
50%,16.100000,6.700000,8.300000
75%,21.950000,13.600000,16.100000
max,26.700000,17.800000,21.700000


We can also provide multiple values to pivot on, which will result in a hierarchical index:

In [ ]:
# values=['temp_C', 'temp_F'] → 값을 2개 주면 열이 2층으로 생김
# (temp_C 아래에 TMAX/TMIN/TOBS, temp_F 아래에 TMAX/TMIN/TOBS)
# 이런 2층 구조를 hierarchical index(계층 인덱스) 또는 MultiIndex(멀티인덱스)라고 함
pivoted_df = long_df.pivot(
    index='date', columns='datatype', values=['temp_C', 'temp_F']
)
pivoted_df.head()


temp_C             temp_F              
datatype     TMAX  TMIN  TOBS   TMAX   TMIN   TOBS
date                                              
2018-10-01   21.1   8.9  13.9  69.98  48.02  57.02
2018-10-02   23.9  13.9  17.2  75.02  57.02  62.96
2018-10-03   25.0  15.6  16.1  77.00  60.08  60.98
2018-10-04   22.8  11.7  11.7  73.04  53.06  53.06
2018-10-05   23.3  11.7  18.9  73.94  53.06  66.02

With the hierarchical index, if we want to select `TMIN` in Fahrenheit, we will first need to select `temp_F` and then `TMIN`:

In [ ]:
# 2층짜리 열에서 값 꺼내기: 바깥층 먼저 ['temp_F'], 그다음 안쪽층 ['TMIN']
# → 화씨 최저기온만 시리즈로
pivoted_df['temp_F']['TMIN'].head()


date
2018-10-01    48.02
2018-10-02    57.02
2018-10-03    60.08
2018-10-04    53.06
2018-10-05    53.06
Name: TMIN, dtype: float64

### `unstack()`

We have been working with a single index throughout this chapter; however, we can create an index from any number of columns with `set_index()`. This gives us an index of type `MultiIndex`, where the outermost level corresponds to the first element in the list provided to `set_index()`:

In [ ]:
# set_index에 열 2개를 리스트로 주면 인덱스가 2층이 됨 → MultiIndex
# 바깥층 date, 안쪽층 datatype. ('2018-10-01', 'TMAX') 처럼 튜플 쌍으로 표시됨
multi_index_df = long_df.set_index(['date', 'datatype'])
multi_index_df.head().index


MultiIndex([('2018-10-01', 'TMAX'),
            ('2018-10-01', 'TMIN'),
            ('2018-10-01', 'TOBS'),
            ('2018-10-02', 'TMAX'),
            ('2018-10-02', 'TMIN')],
           names=['date', 'datatype'])

Notice there are now 2 index sections of the dataframe:

In [ ]:
# 인덱스가 2칸(date, datatype)으로 표시됨. 같은 날짜는 한 번만 쓰고 아래는 비워서 보여줌
multi_index_df.head()


temp_C  temp_F
date       datatype                
2018-10-01 TMAX        21.1   69.98
           TMIN         8.9   48.02
           TOBS        13.9   57.02
2018-10-02 TMAX        23.9   75.02
           TMIN        13.9   57.02

With an index of type `MultiIndex`, we can no longer use `pivot()`. We must now use `unstack()`, which by default moves the innermost index onto the columns:

In [ ]:
# .unstack() → 인덱스의 가장 안쪽 층(datatype)을 열로 올림. 인덱스가 여러 층일 때 쓰는 pivot 같은 거임
# 결과는 위에서 pivot으로 만든 것과 똑같음 (temp_C, temp_F 2층 열)
# stack = 쌓다(열→인덱스로 내리기) / unstack = 쌓은 걸 풀다(인덱스→열로 올리기)
unstacked_df = multi_index_df.unstack()
unstacked_df.head()


temp_C             temp_F              
datatype     TMAX  TMIN  TOBS   TMAX   TMIN   TOBS
date                                              
2018-10-01   21.1   8.9  13.9  69.98  48.02  57.02
2018-10-02   23.9  13.9  17.2  75.02  57.02  62.96
2018-10-03   25.0  15.6  16.1  77.00  60.08  60.98
2018-10-04   22.8  11.7  11.7  73.04  53.06  53.06
2018-10-05   23.3  11.7  18.9  73.94  53.06  66.02

The `unstack()` method also provides the `fill_value` parameter, which let's us fill-in any `NaN` values that might arise from this restructuring of the data. Consider the case that we have data for the average temperature on October 1, 2018, but no other date:

In [ ]:
# 10월 1일에만 TAVG(평균기온) 데이터를 하나 추가해봄 (일부러 빈칸이 생기는 상황 만들기)
# .append([딕셔너리]) → 행 하나 추가 → .set_index(2층) → .sort_index() 날짜순 정렬
# ※ 최신 pandas(2.0 이상)에서는 append()가 없어서 에러남. 대신 이렇게:
#    pd.concat([long_df, pd.DataFrame([{'datatype': 'TAVG', 'date': '2018-10-01', 'temp_C': 10, 'temp_F': 50}])])
extra_data = long_df.append([{
    'datatype': 'TAVG',
    'date': '2018-10-01',
    'temp_C': 10,
    'temp_F': 50
}]).set_index(['date', 'datatype']).sort_index()

# 10/1에는 TAVG가 있고, 10/2에는 없음
extra_data['2018-10-01':'2018-10-02']


temp_C  temp_F
date       datatype                
2018-10-01 TAVG        10.0   50.00
           TMAX        21.1   69.98
           TMIN         8.9   48.02
           TOBS        13.9   57.02
2018-10-02 TMAX        23.9   75.02
           TMIN        13.9   57.02
           TOBS        17.2   62.96

If we use `unstack()` in this case, we will have `NaN` for the `TAVG` columns every day but October 1, 2018:

In [ ]:
# unstack하면 TAVG 열이 생기는데 10/1 말고는 데이터가 없으니 전부 NaN
extra_data.unstack().head()


temp_C                   temp_F                     
datatype     TAVG  TMAX  TMIN  TOBS   TAVG   TMAX   TMIN   TOBS
date                                                           
2018-10-01   10.0  21.1   8.9  13.9   50.0  69.98  48.02  57.02
2018-10-02    NaN  23.9  13.9  17.2    NaN  75.02  57.02  62.96
2018-10-03    NaN  25.0  15.6  16.1    NaN  77.00  60.08  60.98
2018-10-04    NaN  22.8  11.7  11.7    NaN  73.04  53.06  53.06
2018-10-05    NaN  23.3  11.7  18.9    NaN  73.94  53.06  66.02

To address this, we can pass in an appropriate `fill_value`. However, we are restricted to passing in a value for this, not a strategy (like we saw with `fillna()`), so while `-40` is definitely not be the best value, we can use it to illustrate how this works, since this is the temperature at which Fahrenheit and Celsius are equal:

In [ ]:
# fill_value=-40 → unstack하면서 생기는 빈칸(NaN)을 -40으로 채워줘
# (-40은 섭씨=화씨가 같아지는 온도라서 예시로 쓴 것. 실제로 좋은 값은 아님. "이렇게 채울 수 있다"만 보면 됨)
extra_data.unstack(fill_value=-40).head()


temp_C                   temp_F                     
datatype     TAVG  TMAX  TMIN  TOBS   TAVG   TMAX   TMIN   TOBS
date                                                           
2018-10-01   10.0  21.1   8.9  13.9   50.0  69.98  48.02  57.02
2018-10-02  -40.0  23.9  13.9  17.2  -40.0  75.02  57.02  62.96
2018-10-03  -40.0  25.0  15.6  16.1  -40.0  77.00  60.08  60.98
2018-10-04  -40.0  22.8  11.7  11.7  -40.0  73.04  53.06  53.06
2018-10-05  -40.0  23.3  11.7  18.9  -40.0  73.94  53.06  66.02

## Melting
Going from wide to long format.

### Setup

In [ ]:
# 이번엔 wide 형식 데이터 읽기 (wide → long 으로 바꿔볼 거임)
wide_df = pd.read_csv('data/wide_data.csv')
wide_df.head()


,date,TMAX,TMIN,TOBS
0,2018-10-01,21.1,8.9,13.9
1,2018-10-02,23.9,13.9,17.2
2,2018-10-03,25.0,15.6,16.1
3,2018-10-04,22.8,11.7,11.7
4,2018-10-05,23.3,11.7,18.9


### `melt()`
In order to go from wide format to long format, we use the `melt()` method. We have to specify:
- `id_vars`: which column(s) uniquely identify a row in the wide format (`date`, here)
- `value_vars`: the column(s) that contain(s) the values (`TMAX`, `TMIN`, and `TOBS`, here)

Optionally, we can also provide:
- `value_name`: what to call the column that will contain all the values once melted
- `var_name`: what to call the column that will contain the names of the variables being measured

In [ ]:
# .melt() → wide → long으로 (melt = 녹이다. 넓은 표를 녹여서 길게 늘어뜨림). pivot의 반대
# id_vars='date' → 그대로 남겨둘 열 (각 행을 구분하는 기준)
# value_vars=['TMAX', 'TMIN', 'TOBS'] → 이 열들을 녹여서 세로로 늘어뜨림
# value_name='temp_C' → 값들이 들어갈 열 이름
# var_name='measurement' → 원래 열 이름(TMAX 등)이 들어갈 열 이름
# 결과: TMAX 31행, 그다음 TMIN 31행, TOBS 31행 순으로 쌓임 (총 93행)
melted_df = wide_df.melt(
    id_vars='date',
    value_vars=['TMAX', 'TMIN', 'TOBS'],
    value_name='temp_C',
    var_name='measurement'
)
melted_df.head()


,date,measurement,temp_C
0,2018-10-01,TMAX,21.1
1,2018-10-02,TMAX,23.9
2,2018-10-03,TMAX,25.0
3,2018-10-04,TMAX,22.8
4,2018-10-05,TMAX,23.3


### `stack()`
Another option is `stack()`, which will pivot the columns of the dataframe into the innermost level of the index (resulting in an index of type `MultiIndex`). To illustrate this, let's set our index to be the `date` column:

In [ ]:
# stack()을 쓰려면 먼저 date를 인덱스로 (안 그러면 date까지 값으로 취급돼서 섞임)
wide_df.set_index('date', inplace=True)
wide_df.head()


,TMAX,TMIN,TOBS
date,,,
2018-10-01,21.1,8.9,13.9
2018-10-02,23.9,13.9,17.2
2018-10-03,25.0,15.6,16.1
2018-10-04,22.8,11.7,11.7
2018-10-05,23.3,11.7,18.9


By running `stack()` now, we will create a second level in our index which will contain the column names of our dataframe (`TMAX`, `TMIN`, `TOBS`). This will leave us with a `Series` object containing the values:

In [ ]:
# .stack() → 열 이름(TMAX, TMIN, TOBS)을 인덱스의 안쪽 층으로 내림. unstack의 반대
# 열이 하나도 안 남으니 결과는 데이터프레임이 아니라 시리즈 (2층 인덱스 + 값)
stacked_series = wide_df.stack()
stacked_series.head()


date            
2018-10-01  TMAX    21.1
            TMIN     8.9
            TOBS    13.9
2018-10-02  TMAX    23.9
            TMIN    13.9
dtype: float64

We can use the `to_frame()` method on our `Series` object to turn it into a `DataFrame` object. Since the series doesn't have a name at the moment, we will pass in the name as an argument:

In [ ]:
# .to_frame('values') → 시리즈를 데이터프레임으로. 열 이름을 'values'로 정해줌
# (2_creating_dataframes에서 봤던 to_frame)
stacked_df = stacked_series.to_frame('values')
stacked_df.head()


values
date                   
2018-10-01 TMAX    21.1
           TMIN     8.9
           TOBS    13.9
2018-10-02 TMAX    23.9
           TMIN    13.9

Once again, we have an index of type `MultiIndex`:

In [ ]:
# 인덱스가 2층 MultiIndex. names=['date', None] → 안쪽 층은 이름이 없음(None)
stacked_df.head().index


MultiIndex([('2018-10-01', 'TMAX'),
            ('2018-10-01', 'TMIN'),
            ('2018-10-01', 'TOBS'),
            ('2018-10-02', 'TMAX'),
            ('2018-10-02', 'TMIN')],
           names=['date', None])

Unfortunately, we don't have a name for the `datatype` level:

In [ ]:
# .index.names → 인덱스 각 층의 이름. 두 번째가 None(이름 없음)
stacked_df.index.names


FrozenList(['date', None])

We can use `set_names()` to address this though:

In [ ]:
# .set_names(['date', 'datatype'], inplace=True) → 인덱스 층 이름을 정해줌. 이제 둘 다 이름 있음
stacked_df.index.set_names(['date', 'datatype'], inplace=True)
stacked_df.index.names


FrozenList(['date', 'datatype'])

<hr>
<div>
    <a href="./3-cleaning_data.ipynb">
        <button>&#8592; Previous Notebook</button>
    </a>
    <a href="./5-handling_data_issues.ipynb">
        <button style="float: right;">Next Notebook &#8594;</button>
    </a>
</div>
<hr>